# E4: có hay không có `adr` và `total_of_special_requests`

**Câu hỏi:** hai cột "vùng xám" đóng góp bao nhiêu cho mô hình?

- `adr` (giá trung bình mỗi đêm): thường biết lúc đặt, nhưng giá trong dữ liệu là giá cuối cùng nên chưa chắc. Bản chính **giữ** nó.
- `total_of_special_requests`: không xác định được thời điểm ghi nhận, và khách hủy sớm ít có cơ hội gửi thêm yêu cầu. Bản chính **loại** nó, chọn thận trọng.

**Thiết kế:** lưới 2x2 (có/không `adr`) x (có/không `special_requests`), cùng hai mô hình tham chiếu mặc định, cùng một cách chia theo thời gian. Bản có `special_requests` tạo bằng `--keep-special-requests`; việc bỏ `adr` dùng tham số `drop=["adr"]` (cùng dữ liệu, chỉ khác cột đưa vào mô hình). Chỉ dùng `train` và `val`.

**Cách đọc:** nếu thêm một cột làm điểm tăng rõ thì có hai khả năng: cột đó thật sự có ích, hoặc nó làm lộ đáp án vì chỉ biết sau thời điểm đặt. Thực nghiệm này *đo* mức đóng góp nhưng không tự phân biệt được hai khả năng, phần đó phải bằng lập luận.

In [ ]:
import sys, subprocess; sys.path.insert(0, "..")

import pandas as pd

from src.data import REPO_ROOT, TARGET, load_split
from src.split import time_split
from src.reference import make_reference, REFERENCE
from src.evaluate import evaluate, log_result
from src.compare import paired_bootstrap

AUTHOR = "tuan"
REQ = "total_of_special_requests"
SHOW = ["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy"]

RAW = REPO_ROOT / "data" / "raw" / "hotel_bookings.csv"
MAIN = REPO_ROOT / "data" / "interim" / "hotel_bookings_clean.csv"
WITHREQ = REPO_ROOT / "data" / "interim" / "hotel_bookings_withreq.csv"
SCRIPT = REPO_ROOT / "src" / "clean_hotel_bookings.py"

## Tạo bản giữ `total_of_special_requests`

In [ ]:
res = subprocess.run([sys.executable, str(SCRIPT), str(RAW), str(WITHREQ), "--keep-special-requests"],
                     capture_output=True, text=True, check=True)
print(res.stdout)

main = pd.read_csv(MAIN)
withreq = pd.read_csv(WITHREQ)

# kiểm tra: bản mới chỉ hơn bản chính đúng một cột, các dòng và giá trị khác giữ nguyên
assert len(main) == len(withreq)
assert set(withreq.columns) - set(main.columns) == {REQ}
assert withreq[main.columns].equals(main), "bản mới khác bản chính ở các cột còn lại"
print(f"OK: {len(withreq):,} dòng, thêm đúng cột {REQ}")

In [ ]:
parts = time_split(withreq)
for s in ("train", "val"):
    assert len(parts[s]) == len(load_split(s)), f"{s} không khớp notebook 02"

Xtr, ytr = parts["train"].drop(columns=[TARGET]), parts["train"][TARGET]
Xv, yv = parts["val"].drop(columns=[TARGET]), parts["val"][TARGET]
print(f"train {Xtr.shape}, val {Xv.shape}")

Mối quan hệ giữa số yêu cầu đặc biệt và việc hủy, chỉ trên tập huấn luyện:

In [ ]:
(parts["train"].groupby(REQ)[TARGET]
 .agg(so_dong="size", ty_le_huy="mean").round(3))

## Chạy lưới 2x2

In [ ]:
CONFIGS = {
    "default":         dict(drop=[],      extra_numeric=[]),       # có adr, không có special_requests (bản chính)
    "no_adr":          dict(drop=["adr"], extra_numeric=[]),
    "with_req":        dict(drop=[],      extra_numeric=[REQ]),
    "no_adr_with_req": dict(drop=["adr"], extra_numeric=[REQ]),
}

rows, probas = [], {}
for name in REFERENCE:
    for cfg, kw in CONFIGS.items():
        model = make_reference(name, **kw).fit(Xtr, ytr)
        p = model.predict_proba(Xv)[:, 1]
        probas[(name, cfg)] = p
        m = evaluate(yv, p)
        log_result(AUTHOR, "E4", name, "val", {**m, "params": "default"}, note=cfg)
        rows.append({"mô hình": name, "cấu hình": cfg, **m})

e4 = pd.DataFrame(rows).set_index(["mô hình", "cấu hình"])[SHOW].round(3)
e4

## Mỗi cột đóng góp bao nhiêu so với bản chính

Chênh lệch so với `default` trên cùng các dòng kiểm định, kèm khoảng tin cậy 95% (bootstrap ghép cặp). Chênh lệch dương là cấu hình đó *tốt hơn* `default`.

In [ ]:
rows = []
for name in REFERENCE:
    for cfg in ("no_adr", "with_req", "no_adr_with_req"):
        for metric in ("pr_auc", "roc_auc"):
            r = paired_bootstrap(yv, probas[(name, "default")], probas[(name, cfg)], metric)
            rows.append({"mô hình": name, "so với default": cfg, "chỉ số": metric, **r})
pd.DataFrame(rows).set_index(["mô hình", "so với default", "chỉ số"]).round(4)

Khoảng tin cậy chỉ đo nhiễu do chọn mẫu kiểm định, không đo nhiễu do huấn luyện. Khoảng chứa 0 nghĩa là chưa thấy khác biệt rõ, không phải hai cách bằng nhau.

## Kết luận E4 (điền sau khi chạy)

- **`adr`:** bỏ nó làm điểm giảm bao nhiêu, và có lớn hơn nhiễu không? Với mô hình tuyến tính và mô hình cây có giống nhau không?
- **`total_of_special_requests`:** thêm nó làm điểm tăng bao nhiêu? Nếu tăng rõ, đó là thông tin hữu ích hay là rò rỉ (chỉ biết sau khi đặt)? Dựa vào bảng tỷ lệ hủy theo số yêu cầu và lập luận về thời điểm ghi nhận để trả lời.
- Quyết định của nhóm (giữ `adr`, loại `special_requests`) có hợp lý không? Nêu cái giá của việc thận trọng: bỏ `special_requests` làm mất bao nhiêu điểm.
- Nếu thêm `special_requests` mà mô hình tốt lên nhiều, nêu rõ trong báo cáo rằng đó là điểm "có thể không dùng được ngoài đời", và vì vậy nhóm không đưa nó vào mô hình chính.